# Data Loading and Preparation for O*NET Career Analysis

This notebook prepares O*NET occupational data for machine learning analysis. The goal is to combine **Abilities** and **Skills** datasets into a single, structured dataset that can be used for clustering jobs based on their required competencies.

## 1. Import Library

In [10]:
# Data manipulation
import pandas as pd

## 2. Load the Data

We load two separate O*NET datasets:
- **Abilities**: Innate traits that influence how well someone can perform tasks (e.g., "Oral Comprehension", "Arm-Hand Steadiness")
- **Skills**: Learned competencies developed through training or experience (e.g., "Programming", "Critical Thinking")

In [11]:
# Load Abilities dataset from O*NET
df_abilities = pd.read_excel('../Data/ONET/Abilities.xlsx')
print('Abilities')
print(df_abilities.head())

# Load Skills dataset from O*NET
df_skills = pd.read_excel('../Data/ONET/Skills.xlsx')
print('Skills')
print(df_skills.head())

Abilities
  O*NET-SOC Code             Title Element ID           Element Name Scale ID  \
0     11-1011.00  Chief Executives  1.A.1.a.1     Oral Comprehension       IM   
1     11-1011.00  Chief Executives  1.A.1.a.1     Oral Comprehension       LV   
2     11-1011.00  Chief Executives  1.A.1.a.2  Written Comprehension       IM   
3     11-1011.00  Chief Executives  1.A.1.a.2  Written Comprehension       LV   
4     11-1011.00  Chief Executives  1.A.1.a.3        Oral Expression       IM   

   Scale Name  Data Value  N  Standard Error  Lower CI Bound  Upper CI Bound  \
0  Importance        4.62  8          0.1830          4.2664          4.9836   
1       Level        4.88  8          0.1250          4.6300          5.1200   
2  Importance        4.25  8          0.1637          3.9292          4.5708   
3       Level        4.88  8          0.1250          4.6300          5.1200   
4  Importance        4.50  8          0.1890          4.1296          4.8704   

  Recommend Suppress N

## 3. Data Validation

Before merging datasets, we verify that both contain the same job titles.

In [12]:
# Count unique jobs in each dataset
print("Unique job titles abilities: ", df_abilities['Title'].nunique())
print("Unique job titles skills: ", df_skills['Title'].nunique())

# Extract job title sets for comparison
ids_abilities = set(df_abilities['Title'].unique())
ids_skills = set(df_skills['Title'].unique())

# Verify both datasets contain the same jobs
print("Same job titles in both datasets: ", ids_abilities == ids_skills)

Unique job titles abilities:  894
Unique job titles skills:  894
Same job titles in both datasets:  True


## 4. Filter for Level Values (LV)

O*NET provides two types of ratings for each skill/ability:
- **IM (Importance)**: How important is this skill for the job?
- **LV (Level)**: What level of proficiency is required?

In [13]:
# Filter Abilities: keep only Level values ('LV'), not Importance ('IM')
df_abilities = df_abilities.loc[
    df_abilities['Scale ID'] == 'LV',
    ['O*NET-SOC Code', "Title", 'Element Name', 'Data Value']
]

df_abilities.head()

,O*NET-SOC Code,Title,Element Name,Data Value
1,11-1011.00,Chief Executives,Oral Comprehension,4.88
3,11-1011.00,Chief Executives,Written Comprehension,4.88
5,11-1011.00,Chief Executives,Oral Expression,4.88
7,11-1011.00,Chief Executives,Written Expression,4.75
9,11-1011.00,Chief Executives,Fluency of Ideas,4.62


In [14]:
# Filter Skills: same approach as Abilities
skills = df_skills.loc[
    df_skills['Scale ID'] == 'LV',
    ['O*NET-SOC Code', "Title", 'Element Name', 'Data Value']
]

skills.head()

,O*NET-SOC Code,Title,Element Name,Data Value
1,11-1011.00,Chief Executives,Reading Comprehension,4.62
3,11-1011.00,Chief Executives,Active Listening,4.75
5,11-1011.00,Chief Executives,Writing,4.38
7,11-1011.00,Chief Executives,Speaking,4.75
9,11-1011.00,Chief Executives,Mathematics,3.50


## 5. Add Prefixes to Distinguish Features

Since both datasets have an "Element Name" column (the name of each skill or ability), we add prefixes:
- **"A: "** for Abilities
- **"S: "** for Skills

This prevents column name conflicts when merging and makes it easy to identify the source of each feature in the final dataset.

In [15]:
df_abilities['Element Name'] = "A: " + df_abilities['Element Name']
df_skills['Element Name']    = "S: " + df_skills['Element Name']

## 6. Pivot Tables: Long to Wide Format

The original data is in "long format": one row per job-skill combination. For machine learning, we need "wide format": one row per job, with each skill/ability as its own column.

In [16]:
# Pivot Skills: transform from long to wide format
skills_wide = df_skills.pivot_table(
    index='O*NET-SOC Code',      # Rows: unique job codes
    columns='Element Name',      # Columns: skill names
    values='Data Value'          # Cell values: proficiency levels
)

# Pivot Abilities: same transformation
# Include Title as index to preserve job names for later reference
abilities_wide = df_abilities.pivot_table(
    index=['O*NET-SOC Code', 'Title'],  # Multi-index to keep job title
    columns='Element Name',
    values='Data Value'
)

## 7. Merge Datasets

We join the Skills and Abilities datasets using the **O\*NET-SOC Code** (unique job identifier). This creates one comprehensive dataset with all features for each occupation.

**fillna(0)**: Any missing values are filled with 0, assuming that if a skill/ability isn't rated for a job, it's not required (level = 0).

In [17]:
# Join Abilities and Skills into one dataset using SOC Code as the key
full_df = abilities_wide.join(skills_wide, on='O*NET-SOC Code')

# Handle missing values: fill with 0 (no skill required = level 0)
full_df = full_df.fillna(0)

# Verify the result
print("Shape:", full_df.shape) 
full_df.head()

Shape: (894, 87)


,Element Name,A: Arm-Hand Steadiness,A: Auditory Attention,A: Category Flexibility,A: Control Precision,A: Deductive Reasoning,A: Depth Perception,A: Dynamic Flexibility,A: Dynamic Strength,A: Explosive Strength,A: Extent Flexibility,...,S: Science,S: Service Orientation,S: Social Perceptiveness,S: Speaking,S: Systems Analysis,S: Systems Evaluation,S: Technology Design,S: Time Management,S: Troubleshooting,S: Writing
O*NET-SOC Code,Title,,,,,,,,,,,,,,,,,,,,,
11-1011.00,Chief Executives,0.50,2.00,4.00,0.75,4.75,1.38,0.0,0.25,0.00,0.00,...,1.185,3.250,4.185,4.50,4.62,4.625,1.315,4.375,1.000,4.250
11-1011.03,Chief Sustainability Officers,0.00,1.75,3.50,0.50,4.75,1.75,0.0,0.00,0.00,0.00,...,2.000,3.250,3.880,4.06,3.94,3.940,1.500,3.630,0.500,4.185
11-1021.00,General and Operations Managers,0.88,2.00,3.25,0.12,4.12,1.62,0.0,0.12,0.38,0.25,...,1.060,3.185,3.875,4.06,3.12,3.185,1.060,3.750,1.375,3.690
11-2011.00,Advertising and Promotions Managers,0.50,1.25,3.88,0.12,4.50,1.00,0.0,0.00,0.00,0.00,...,1.120,3.185,4.000,4.06,3.12,3.435,1.250,3.690,0.500,3.815
11-2021.00,Marketing Managers,0.12,1.62,3.62,0.00,4.38,0.75,0.0,0.25,0.00,0.00,...,1.625,3.185,3.940,4.00,3.50,3.625,1.315,3.625,0.500,3.565


## 8. Export Prepared Data

The final dataset is saved to CSV for use in the machine learning notebooks. This file contains:
- One row per occupation
- Columns for each Ability (prefixed "A:") and Skill (prefixed "S:")
- Values representing the required proficiency level (0-7 scale)

In [18]:
# Save the prepared dataset for use in ML clustering notebooks
full_df.to_csv('../Data/career_pivot_results.csv')

### Output File
`career_pivot_results.csv` — Ready for use in clustering notebooks


### Next Steps
→ Proceed to K-means PCA notebook